In [ ]:
#!pip install jupyter_dash dash plotly
#!pip install jupyter_dash==0.4.2 dash==2.11.1
#!C:\Users\alfre\AppData\Local\Programs\Python\Python312\python.exe -m pip install dash==2.11.1 jupyter_dash==0.4.2

In [ ]:
import dash
import jupyter_dash
print("Dash version:", dash.__version__)
print("JupyterDash version:", jupyter_dash.__version__)

In [9]:
from IPython.display import clear_output
clear_output(wait=True)

import dash
from dash import dcc, html
from dash.dependencies import Input, Output, State
import plotly.graph_objs as go

# -----------------------------
# Node positions
# -----------------------------
nodes = {
    "Glucose": (0, 1),
    "Fat": (0, -1),
    "Citrate": (1, 0),
    "Malonyl": (2, 0)
}

# -----------------------------
# Global state
# -----------------------------
G = F = C = M = 0.0

# -----------------------------
# Metabolic step
# -----------------------------
def step(glucose_target, fat_target, G, F, C, M):
    # simple relaxation dynamics
    G += 0.05 * (glucose_target - G) - 0.2 * C
    F += 0.05 * (fat_target - F)

    # simple feedback loops
    C += 0.05 * (G - C)
    M += 0.05 * (C - M)

    return G, F, C, M

# -----------------------------
# Dash app
# -----------------------------
app = dash.Dash(__name__)

app.layout = html.Div([
    dcc.Slider(id='glucose', min=0, max=100, value=50, updatemode='drag'),
    dcc.Slider(id='fat', min=0, max=100, value=50, updatemode='drag'),

    dcc.Graph(id='network'),

    dcc.Interval(id='tick', interval=50, n_intervals=0)
])

def flux_color(value):
    # value is between 0 and 1
    value_adj = value ** 0.8   # gamma correction
    r = int(255 * value_adj)
    b = int(255 * (1 - value_adj))
    return f"rgb({r},0,{b})"

# -----------------------------
# Animation callback
# -----------------------------
@app.callback(
    Output('network', 'figure'),
    Input('tick', 'n_intervals'),
    State('glucose', 'value'),
    State('fat', 'value')
)
def animate(_, glucose_target, fat_target):
    global G, F, C, M

    # evolve system
    G, F, C, M = step(glucose_target, fat_target, G, F, C, M)

    # build figure
    fig = go.Figure()

    # draw nodes
    for name, (x, y) in nodes.items():
        fig.add_trace(go.Scatter(
            x=[x], y=[y],
            mode='markers+text',
            text=[name],
            marker=dict(size=20)
        ))

    # draw edges with thickness = flux
    def edge(a, b, flux_value):
        x1, y1 = nodes[a]
        x2, y2 = nodes[b]
    
        # normalize flux to 0–1 range
        flux_norm = max(0, min(1, flux_value))
    
        # compute color
        color = flux_color(flux_norm)
    
        fig.add_trace(go.Scatter(
            x=[x1, x2],
            y=[y1, y2],
            mode='lines',
            line=dict(
                width=flux_value * 1.2,
                color=color
            ),
            hoverinfo='none'
        ))

    edge("Glucose", "Citrate", G/10)
    edge("Fat", "Citrate", F/10)
    edge("Citrate", "Malonyl", C/10)
    edge("Malonyl", "Fat", M/10)

    fig.update_layout(showlegend=False)
    #fig.update_xaxes(visible=False)
    #fig.update_yaxes(visible=False)
    
    fig.update_xaxes(
        range=[-1, 3],
        showgrid=False,
        showticklabels=False,
        zeroline=False
    )
    
    fig.update_yaxes(
        range=[-2, 2],
        showgrid=False,
        showticklabels=False,
        zeroline=False
    )



    return fig


print("Dash app running at: http://127.0.0.1:8050/")
app.run_server(mode='external')




Dash app running at: http://127.0.0.1:8050/
